# SpaceInvasion GA + Neural Network Optimizer — Experimentation Notebook

This interactive notebook provides a self-contained, read-only-safe laboratory for exploring and tuning the `spaceinvasion` Genetic Algorithm + Neural Network (`ga_nn`) action-combination optimizer:

1. **Environment & Complete History Data Loading**: CWD-agnostic path resolution, read-only loading of `data/attack_history.json` + WAL-safe temporary-copy inspection of `state/spaceinvasion.db`, and a seeded `:memory:` `GameDatabase` exercising complete multi-table telemetry (`raid_history`, `spy_targets`, `planet_snapshots`, `rank_snapshots`, `action_history`).
2. **Feature Engineering & Neural Value Network Evaluation**: Inspecting the 46-dimensional empire & raid-history feature vector (`encode_empire_state`), comparing cold-start vs complete-history representations, training `PurePythonMLP` / optional `TorchMLP` on simulated rollouts (`TrainingStats`), and computing input sensitivities (`dV/dx_j`).
3. **GA + NN Action Search, Baseline Comparison & Hyperparameter Tuning**: Running `run_optimizer` / `optimize_plan_actions`, inspecting generation-by-generation `OptimizerRunSummary.generation_history`, comparing `"priority"` vs `"ga_nn"` plans with Queue ETAs (`plan_actions` / `attach_action_etas`), and running a fast interactive hyperparameter tuning sweep.

In [ ]:
import contextlib
from dataclasses import replace
import hashlib
import json
import math
from os import environ
from pathlib import Path
import shutil
import sqlite3
import sys
import tempfile
import time
from typing import Any


def resolve_repo_root() -> Path:
    """Locate repository root without relying on `__file__`."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "spaceinvasion" / "__init__.py").is_file():
            return cand
    for entry in list(sys.path):
        if not entry:
            continue
        p = Path(entry).resolve()
        if (p / "spaceinvasion" / "__init__.py").is_file():
            return p
    import spaceinvasion  # raises ImportError if project is unreachable
    return Path(spaceinvasion.__file__).resolve().parents[1]


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer isolated `cwd` data/state fixtures when present; else use repo root."""
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "attack_history.json").is_file() or (
        cwd / "state" / "spaceinvasion.db"
    ).is_file():
        return cwd
    return repo_root


ACTIVE_DATA_ROOT = resolve_active_data_root(REPO_ROOT)
ATTACK_HISTORY_PATH = ACTIVE_DATA_ROOT / "data" / "attack_history.json"
LIVE_DB_PATH = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"

from spaceinvasion import game_data
from spaceinvasion.config import StrategyConfig
from spaceinvasion.fleet import SpyReport
from spaceinvasion.logging_utils import redact_sensitive
from spaceinvasion.optimizer import (
    STATE_FEATURE_DIM,
    TORCH_AVAILABLE,
    GenerationStats,
    NeuralValueNetwork,
    OptimizerRunSummary,
    PurePythonMLP,
    SimAction,
    TorchMLP,
    create_value_network,
    encode_empire_state,
    load_attack_history_stats,
    optimize_plan_actions,
    run_optimizer,
    simulate_action_sequence,
)
from spaceinvasion.ranking import (
    attach_action_etas,
    compute_action_eta,
    estimate_account_points,
    estimate_hourly_economy,
)
from spaceinvasion.state import (
    BuildingOffer,
    EnergyBalance,
    GameState,
    PlanetState,
    ResearchOffer,
    Resources,
    UnitOffer,
)
from spaceinvasion.storage import GameDatabase
from spaceinvasion.strategy import PlannedAction, plan_actions

try:
    import torch  # type: ignore[import-not-found]
    TORCH_VERSION: str | None = str(getattr(torch, "__version__", "installed"))
except ImportError:
    torch = None  # type: ignore[assignment]
    TORCH_VERSION = None

try:
    _headless_mpl = "ipykernel" not in sys.modules and "MPLBACKEND" not in environ
    if _headless_mpl:
        environ.setdefault("MPLBACKEND", "Agg")
    import matplotlib  # type: ignore[import-not-found]
    if _headless_mpl:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt  # type: ignore[import-not-found]
    HAS_MATPLOTLIB = True
except Exception:
    matplotlib = None  # type: ignore[assignment]
    plt = None  # type: ignore[assignment]
    HAS_MATPLOTLIB = False


def render_table(headers: list[str], rows: list[list[Any]]) -> str:
    str_rows = [[str(cell) for cell in row] for row in rows]
    widths = [len(h) for h in headers]
    for row in str_rows:
        for idx, cell in enumerate(row):
            if idx < len(widths):
                widths[idx] = max(widths[idx], len(cell))
    sep = "+-" + "-+-".join("-" * w for w in widths) + "-+"
    head_line = "| " + " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers)) + " |"
    lines = [sep, head_line, sep]
    for row in str_rows:
        padded = [
            (row[i] if i < len(row) else "").ljust(widths[i])
            for i in range(len(widths))
        ]
        lines.append("| " + " | ".join(padded) + " |")
    lines.append(sep)
    return "\n".join(lines)


def ascii_bar(value: float, max_value: float, width: int = 20) -> str:
    if (
        not math.isfinite(max_value)
        or max_value <= 0.0
        or not math.isfinite(value)
        or value <= 0.0
    ):
        return "[" + ("." * width) + "]"
    ratio = max(0.0, min(1.0, float(value) / float(max_value)))
    filled = int(round(ratio * width))
    return "[" + ("#" * filled) + ("." * (width - filled)) + "]"


print("Environment Summary")
print(
    render_table(
        ["Setting", "Value"],
        [
            ["REPO_ROOT", str(REPO_ROOT)],
            ["ACTIVE_DATA_ROOT", str(ACTIVE_DATA_ROOT)],
            ["ATTACK_HISTORY_PATH", f"{ATTACK_HISTORY_PATH} (exists={ATTACK_HISTORY_PATH.is_file()})"],
            ["LIVE_DB_PATH", f"{LIVE_DB_PATH} (exists={LIVE_DB_PATH.is_file()})"],
            ["STATE_FEATURE_DIM", STATE_FEATURE_DIM],
            ["TORCH_AVAILABLE", f"{TORCH_AVAILABLE} ({TORCH_VERSION or 'PurePythonMLP fallback'})"],
            ["HAS_MATPLOTLIB", HAS_MATPLOTLIB],
        ],
    )
)

## 1. Complete Attack & Telemetry History Loading

To avoid mutating the tracked `data/attack_history.json` file or touching the live SQLite WAL database (`state/spaceinvasion.db`), this section:
- Reads `ATTACK_HISTORY_PATH` via `load_attack_history_stats(...)` with defensive `.get(...)` defaults for cold-start JSON payloads.
- Copies any existing SQLite database and active `-wal`/`-shm`/`-journal` sidecars into a `tempfile.TemporaryDirectory()` before opening `GameDatabase` with an isolated mirror path, closing it immediately afterward so the original database timestamps and files remain untouched and no player PII (`nickname`, `owner`) or session tokens are printed.
- Seeds an isolated `:memory:` `GameDatabase` (`attack_history_path=None`) across all telemetry tables (`raid_history`, `spy_targets`, `planet_snapshots`, `rank_snapshots`, `action_history`) so experiments demonstrate both empirical win-rate aggregation (`9/10` vs `1/10` fluke with identical last-attack loot) and multi-snapshot telemetry trends.

In [ ]:
def _as_dict(val: Any) -> dict[str, Any]:
    return dict(val) if isinstance(val, dict) else {}


def _as_list(val: Any) -> list[Any]:
    return list(val) if isinstance(val, list) else []


def _safe_float(val: Any, default: float = 0.0) -> float:
    if isinstance(val, bool) or val is None:
        return default
    try:
        out = float(val)
    except (TypeError, ValueError):
        return default
    return out if math.isfinite(out) else default


def _safe_int(val: Any, default: int = 0) -> int:
    if isinstance(val, bool) or val is None:
        return default
    try:
        if isinstance(val, str):
            return int(float(val))
        return int(val)
    except (TypeError, ValueError, OverflowError):
        return default


disk_attack_history = _as_dict(load_attack_history_stats(ATTACK_HISTORY_PATH))
attack_history_sha256: str | None = None
try:
    if ATTACK_HISTORY_PATH.is_file():
        attack_history_sha256 = hashlib.sha256(ATTACK_HISTORY_PATH.read_bytes()).hexdigest()
except OSError:
    attack_history_sha256 = None

disk_overall = _as_dict(disk_attack_history.get("overall"))
disk_per_target = {
    str(k): _as_dict(v)
    for k, v in _as_dict(disk_attack_history.get("per_target")).items()
}
disk_history_rows = [
    _as_dict(row) for row in _as_list(disk_attack_history.get("history")) if isinstance(row, dict)
]
disk_telemetry = _as_dict(disk_attack_history.get("telemetry_summary"))
disk_tel_planet_snaps = _as_dict(disk_telemetry.get("planet_snapshots"))
disk_tel_rank_snaps = _as_dict(disk_telemetry.get("rank_snapshots"))
disk_tel_action_hist = _as_dict(disk_telemetry.get("action_history"))
disk_tel_spy_targets = _as_dict(disk_telemetry.get("spy_targets"))
disk_tel_raid_hist = _as_dict(disk_telemetry.get("raid_history"))
disk_last_atk = _as_dict(disk_overall.get("last_attack"))
disk_last_loot_sp = _safe_float(disk_last_atk.get("loot_spice_eq"), 0.0)
disk_action_cnt = _safe_int(
    disk_telemetry.get("action_count"),
    _safe_int(disk_tel_action_hist.get("total"), 0),
)
disk_planet_snap_cnt = _safe_int(
    disk_telemetry.get("planet_snapshots_count"),
    _safe_int(disk_tel_planet_snaps.get("count"), 0),
)
disk_rank_snap_cnt = _safe_int(
    disk_telemetry.get("rank_snapshots_count"),
    _safe_int(disk_tel_rank_snaps.get("count"), 0),
)
disk_spy_cnt = _safe_int(
    disk_telemetry.get("spy_targets_count"),
    _safe_int(disk_tel_spy_targets.get("count"), 0),
)
disk_raid_tel_cnt = _safe_int(
    disk_tel_raid_hist.get("count"),
    _safe_int(disk_tel_raid_hist.get("attempts"), len(disk_history_rows)),
)

print("Read-Only Tracked Attack History JSON Summary")
print(
    render_table(
        ["Metric", "Value"],
        [
            ["SHA-256", attack_history_sha256 or "N/A (file absent)"],
            ["Attempts / Wins / Losses", f"{_safe_int(disk_overall.get('attempts'))} / {_safe_int(disk_overall.get('wins'))} / {_safe_int(disk_overall.get('losses'))}"],
            ["Success Rate", f"{_safe_float(disk_overall.get('success_rate')):.2%}"],
            ["Total Loot (SpiceEq)", f"{_safe_float(disk_overall.get('total_loot_spice_eq')):,.1f}"],
            ["Avg / Recency SpiceEq", f"{_safe_float(disk_overall.get('avg_loot_spice_eq')):,.1f} / {_safe_float(disk_overall.get('recency_weighted_loot_spice_eq')):,.1f}"],
            ["Last Attack Loot (SpiceEq)", f"{disk_last_loot_sp:,.1f}"],
            ["Tracked Targets / History Rows", f"{len(disk_per_target)} / {len(disk_history_rows)} (tel_raids={disk_raid_tel_cnt}, spy={disk_spy_cnt})"],
            ["Telemetry Actions / Snapshots", f"{disk_action_cnt} actions, {disk_planet_snap_cnt} planet snaps, {disk_rank_snap_cnt} rank snaps"],
        ],
    )
)


_EXPECTED_DB_TABLES: dict[str, set[str]] = {
    "meta": {"key", "value", "updated_ts"},
    "rank_snapshots": {
        "id", "ts", "server_points", "server_rank", "estimated_points",
        "pts_per_hr", "spice_value", "planets",
    },
    "planet_snapshots": {
        "id", "ts", "planet_id", "coords", "planet_type", "fields_used",
        "fields_max", "pig_iron", "metal", "kryptonite", "spice",
        "energy_produced", "energy_used", "buildings_json", "ships_json",
        "economy_rates_json",
    },
    "economy_rates_cache": {"planet_id", "coords", "rates_json", "updated_ts"},
    "tutorial_progress": {
        "step", "total_steps", "tasks_json", "reward_text", "can_complete",
        "first_seen_ts", "last_seen_ts", "completed_ts",
    },
    "action_history": {
        "id", "ts", "source", "command", "priority", "kind", "planet_coords",
        "target_name", "target_level", "executed", "dry_run", "message",
    },
    "spy_targets": {
        "coords", "owner", "is_inactive", "is_spacia", "points", "pig_iron",
        "metal", "kryptonite", "spice", "fleet_json", "defense_json", "dug_in",
        "last_scanned_ts",
    },
    "galaxy_scans": {"galaxy", "system", "last_scanned_ts"},
    "raid_history": {
        "id", "target_coords", "attacked_ts", "is_exempt", "outcome", "loot_json",
    },
}
_EXPECTED_DB_INDEXES: dict[str, str] = {
    "idx_rank_snapshots_ts": "rank_snapshots",
    "idx_planet_snapshots_pid_ts": "planet_snapshots",
    "idx_action_history_ts": "action_history",
    "idx_galaxy_scans_ts": "galaxy_scans",
    "idx_raid_history_coords_ts": "raid_history",
}
_SPY_MIGRATED_COLS: tuple[str, ...] = ("planet_name", "status_color", "last_spied_ts")


def inspect_sqlite_readonly(db_path: Path) -> dict[str, Any]:
    """Copy DB (+ WAL/SHM/journal) into a temp dir, run 4-layer preflight, and query via GameDatabase."""
    out: dict[str, Any] = {
        "exists": False,
        "size_bytes": 0,
        "counts": {},
        "raid_stats": {
            "overall": {"attempts": 0, "wins": 0, "losses": 0, "success_rate": 0.0},
            "per_target": {},
            "history": [],
            "telemetry_summary": {},
        },
        "full_history_summary": {},
        "rank_samples_tail": [],
        "recent_actions_sanitized": [],
        "error": None,
    }
    try:
        exists = db_path.is_file()
        out["exists"] = exists
        if not exists:
            return out
        size_bytes = db_path.stat().st_size
        out["size_bytes"] = size_bytes
        if size_bytes == 0:
            return out
        with tempfile.TemporaryDirectory(prefix="si_nb_ro_") as tmp_dir:
            tmp_root = Path(tmp_dir)
            tmp_db_path = tmp_root / db_path.name
            shutil.copy2(db_path, tmp_db_path)
            tmp_db_path.chmod(0o600)
            for suffix in ("-wal", "-shm", "-journal"):
                sidecar = Path(f"{db_path}{suffix}")
                if sidecar.is_file():
                    tmp_sidecar = Path(f"{tmp_db_path}{suffix}")
                    shutil.copy2(sidecar, tmp_sidecar)
                    tmp_sidecar.chmod(0o600)

            tmp_db_size = tmp_db_path.stat().st_size
            if tmp_db_size < 100:
                raise sqlite3.DatabaseError("file is not a database")

            with contextlib.closing(sqlite3.connect(tmp_db_path)) as probe_conn:
                # Layer 1: Physical alignment & full multi-page B-tree/index/constraint integrity.
                page_size_row = probe_conn.execute("PRAGMA page_size;").fetchone()
                page_size = int(page_size_row[0]) if page_size_row else 0
                if page_size <= 0 or tmp_db_size % page_size != 0:
                    raise sqlite3.DatabaseError("truncated or misaligned SQLite database file")
                ic_rows = probe_conn.execute("PRAGMA integrity_check;").fetchall()
                if [tuple(r) for r in ic_rows] != [("ok",)]:
                    msg = ic_rows[0][0] if ic_rows and ic_rows[0] else "integrity check failed"
                    raise sqlite3.DatabaseError(f"SQLite integrity_check failed: {msg}")

                # Layer 2: Case-insensitive sqlite_master schema, index, column & casing validation.
                schema_rows = probe_conn.execute(
                    "SELECT name, type, tbl_name, sql FROM sqlite_master WHERE type != 'trigger'"
                ).fetchall()
                objects_by_name = {
                    str(name).lower(): (
                        str(name),
                        str(obj_type).lower(),
                        str(tbl_name).lower(),
                        str(sql or ""),
                    )
                    for name, obj_type, tbl_name, sql in schema_rows
                }
                for tbl_name, req_cols in _EXPECTED_DB_TABLES.items():
                    if tbl_name not in objects_by_name:
                        continue
                    orig_name, obj_type, _, obj_sql = objects_by_name[tbl_name]
                    if obj_type != "table" or obj_sql.lstrip().upper().startswith(
                        "CREATE VIRTUAL TABLE"
                    ):
                        raise sqlite3.DatabaseError(
                            f"Incompatible SQLite schema object {tbl_name!r}: type={obj_type!r}"
                        )
                    safe_ident = orig_name.replace('"', '""')
                    cols_exact = {
                        str(row[1])
                        for row in probe_conn.execute(
                            f'PRAGMA table_info("{safe_ident}");'
                        ).fetchall()
                    }
                    cols_lower = {c.lower() for c in cols_exact}
                    missing_cols = sorted(req_cols - cols_lower)
                    if missing_cols:
                        raise sqlite3.DatabaseError(
                            f"Incompatible SQLite table {tbl_name!r}: missing columns {missing_cols}"
                        )
                    if tbl_name == "spy_targets":
                        for mcol in _SPY_MIGRATED_COLS:
                            if mcol in cols_lower and mcol not in cols_exact:
                                raise sqlite3.DatabaseError(
                                    f"Incompatible SQLite column casing for {mcol!r} in spy_targets"
                                )
                for idx_name, expected_tbl in _EXPECTED_DB_INDEXES.items():
                    if idx_name not in objects_by_name:
                        continue
                    _, obj_type, tbl_name, _ = objects_by_name[idx_name]
                    if obj_type != "index" or tbl_name != expected_tbl:
                        raise sqlite3.DatabaseError(
                            f"Incompatible SQLite index object {idx_name!r}: "
                            f"type={obj_type!r}, tbl={tbl_name!r}"
                        )

                # Layer 3: Write/DDL simulation matching GameDatabase.__init__.
                probe_conn.execute("PRAGMA journal_mode=WAL;").fetchall()
                probe_conn.execute("PRAGMA synchronous=NORMAL;")
                probe_conn.execute("PRAGMA foreign_keys=ON;")
                if "spy_targets" in objects_by_name:
                    spy_ident = objects_by_name["spy_targets"][0].replace('"', '""')
                    spy_cols_exact = {
                        str(row[1])
                        for row in probe_conn.execute(
                            f'PRAGMA table_info("{spy_ident}");'
                        ).fetchall()
                    }
                    if "planet_name" not in spy_cols_exact:
                        probe_conn.execute(
                            f'ALTER TABLE "{spy_ident}" ADD COLUMN planet_name TEXT NOT NULL DEFAULT \'\''
                        )
                    if "status_color" not in spy_cols_exact:
                        probe_conn.execute(
                            f'ALTER TABLE "{spy_ident}" ADD COLUMN status_color TEXT NOT NULL DEFAULT \'yellow\''
                        )
                    if "last_spied_ts" not in spy_cols_exact:
                        probe_conn.execute(
                            f'ALTER TABLE "{spy_ident}" ADD COLUMN last_spied_ts REAL NOT NULL DEFAULT 0.0'
                        )
                    if probe_conn.in_transaction:
                        probe_conn.commit()
                if "meta" in objects_by_name:
                    meta_ident = objects_by_name["meta"][0].replace('"', '""')
                    probe_conn.execute("SAVEPOINT _si_preflight_meta;")
                    probe_conn.execute(
                        f'INSERT OR IGNORE INTO "{meta_ident}"(key, value, updated_ts) VALUES(?, ?, ?)',
                        ("schema_version", "1", time.time()),
                    )
                    probe_conn.execute("ROLLBACK TO SAVEPOINT _si_preflight_meta;")
                    probe_conn.execute("RELEASE SAVEPOINT _si_preflight_meta;")
                if probe_conn.in_transaction:
                    probe_conn.rollback()

            # Layer 4: Construct GameDatabase on verified temp copy with atomic out staging.
            ro_db = GameDatabase(
                tmp_db_path,
                attack_history_path=tmp_root / "ro_mirror.json",
            )
            try:
                summary = _as_dict(ro_db.summary())
                counts_val = _as_dict(summary.get("counts"))
                raid_stats_val = redact_sensitive(ro_db.get_raid_stats())
                full_summary_val = redact_sensitive(ro_db.get_full_history_summary())
                rank_tail_val = redact_sensitive(ro_db.get_rank_samples(limit=5))
                clean_actions: list[dict[str, Any]] = []
                for row in ro_db.get_recent_actions(limit=5):
                    row_d = _as_dict(row)
                    tname = row_d.get("target_name")
                    if isinstance(tname, (bytes, bytearray)):
                        tname = tname.decode("utf-8", errors="replace")
                    clean_actions.append(
                        {
                            "ts": row_d.get("ts"),
                            "kind": row_d.get("kind"),
                            "priority": row_d.get("priority"),
                            "planet_coords": row_d.get("planet_coords"),
                            "target_name": redact_sensitive(tname),
                            "executed": bool(row_d.get("executed")),
                            "dry_run": bool(row_d.get("dry_run")),
                        }
                    )
                out["counts"] = counts_val
                out["raid_stats"] = raid_stats_val
                out["full_history_summary"] = full_summary_val
                out["rank_samples_tail"] = rank_tail_val
                out["recent_actions_sanitized"] = clean_actions
            finally:
                ro_db.close()
    except (
        sqlite3.Error,
        OSError,
        ValueError,
        TypeError,
        OverflowError,
        KeyError,
        AttributeError,
    ) as exc:
        out["error"] = redact_sensitive(str(exc)) or type(exc).__name__
    return out


live_db_inspection = inspect_sqlite_readonly(LIVE_DB_PATH)
live_counts = _as_dict(live_db_inspection.get("counts"))
live_full_sum = _as_dict(live_db_inspection.get("full_history_summary"))
print("\nRead-Only Temp-Copy SQLite DB Telemetry")
print(
    render_table(
        ["Table / Metric", "Value"],
        [
            ["DB File Present / Size", f"{live_db_inspection['exists']} / {live_db_inspection['size_bytes']:,} bytes"],
            ["Read Error", live_db_inspection["error"] or "None"],
            ["raid_history / spy_targets", f"{_safe_int(live_counts.get('raid_history'))} / {_safe_int(live_counts.get('spy_targets'))}"],
            ["planet_snapshots / rank_snapshots", f"{_safe_int(live_counts.get('planet_snapshots'))} / {_safe_int(live_counts.get('rank_snapshots'))}"],
            ["action_history / galaxy_scans", f"{_safe_int(live_counts.get('action_history'))} / {_safe_int(live_counts.get('galaxy_scans'))}"],
            ["Points Delta / Pts-per-hr", f"{_safe_float(live_full_sum.get('points_delta')):.2f} / {_safe_float(live_full_sum.get('pts_per_hr')):.2f}"],
        ],
    )
)


def _res_from_cost(raw_cost: dict[str, Any]) -> Resources:
    return Resources(
        pig_iron=int(raw_cost.get("pig_iron", 0) or 0),
        metal=int(raw_cost.get("metal", 0) or 0),
        kryptonite=int(raw_cost.get("kryptonite", 0) or 0),
        spice=int(raw_cost.get("spice", 0) or 0),
    )


def build_sample_game_state(
    *,
    timestamp: float = 1_700_000_000.0,
    resources: Resources = Resources(9_000, 7_000, 4_500, 2_200),
    buildings: dict[str, int] | None = None,
    ships: dict[str, int] | None = None,
    researches: dict[str, int] | None = None,
    include_raid_targets: bool = True,
) -> GameState:
    bld = dict(
        buildings
        if buildings is not None
        else {
            "b1": 7,
            "b2": 5,
            "b3": 6,
            "b4": 4,
            "b5": 10,
            "b7": 3,
            "b8": 2,
            "b9": 3,
            "b11": 3,
            "b12": 3,
            "b13": 3,
            "b14": 3,
        }
    )
    res_tech = dict(
        researches
        if researches is not None
        else {"r1": 2, "r2": 2, "r4": 2, "r7": 2, "r8": 1}
    )
    shp = dict(
        ships
        if ships is not None
        else {"s1": 12, "s4": 28, "s6": 4, "s9": 4}
    )

    e_prod = 0
    e_used = 0
    for bkey, lvl in bld.items():
        if bkey.startswith("b") and bkey[1:].isdigit() and lvl > 0:
            eval_e = int(
                game_data.building_level_stats(int(bkey[1:]), int(lvl)).get("energy", 0)
                or 0
            )
            if eval_e >= 0:
                e_prod += eval_e
            else:
                e_used += -eval_e
    e_prod_eff = int(round(e_prod * (1.0 + 0.01 * int(res_tech.get("r2", 0)))))

    b_offers: list[BuildingOffer] = []
    for bid in (1, 2, 3, 4, 5, 7, 8, 9, 11, 12, 13, 14):
        info = game_data.BUILDING_BY_ID.get(bid) or {}
        cur = int(bld.get(f"b{bid}", 0))
        nxt = cur + 1
        cost = _res_from_cost(game_data.building_cost(bid, nxt))
        unmet = game_data.missing_prerequisites(
            "buildings", bid, buildings=bld, researches=res_tech
        )
        can_b = (not unmet) and resources.can_afford(cost)
        status_b = (
            f"Build level {nxt}"
            if can_b
            else ("Locked" if unmet else "Missing resources")
        )
        b_offers.append(
            BuildingOffer(
                building_id=bid,
                name=str(info.get("name", f"Building {bid}")),
                current_level=cur,
                target_level=nxt,
                cost=cost,
                can_build=can_b,
                status_text=status_b,
                build_href=f"indexInternal.es?action=internalBuildItem&buildingID={bid}",
            )
        )

    ship_specs = (
        (1, "small-transport"),
        (4, "small-fighter"),
        (6, "starcruiser"),
        (9, "solar-cells"),
    )
    s_offers: list[UnitOffer] = []
    for uid, slug in ship_specs:
        uinfo = game_data.SHIPS.get(slug) or {}
        cost = _res_from_cost(uinfo.get("base_cost") or {})
        unmet_s = game_data.missing_prerequisites(
            "ships", slug, buildings=bld, researches=res_tech
        )
        can_s = (not unmet_s) and resources.can_afford(cost)
        status_s = (
            "Buildable"
            if can_s
            else ("Locked" if unmet_s else "Missing resources")
        )
        s_offers.append(
            UnitOffer(
                unit_type="ship",
                unit_id=uid,
                name=str(uinfo.get("name", slug)),
                owned=int(shp.get(f"s{uid}", 0)),
                cost=cost,
                can_build=can_s,
                input_name=f"ship[{uid}]",
                status_text=status_s,
            )
        )

    r_offers: list[ResearchOffer] = []
    for rid, rkey in ((1, "r1"), (2, "r2"), (4, "r4"), (8, "r8")):
        rinfo = game_data.RESEARCH_BY_ID.get(rkey) or {}
        cur = int(res_tech.get(rkey, 0))
        nxt = cur + 1
        lvl_data = (rinfo.get("levels") or {}).get(nxt) or {}
        cost = _res_from_cost(lvl_data.get("cost") or {})
        unmet_r = game_data.missing_prerequisites(
            "research", rkey, buildings=bld, researches=res_tech
        )
        can_r = (not unmet_r) and (not cost.is_zero()) and resources.can_afford(cost)
        status_r = (
            "Ready"
            if can_r
            else ("Locked" if unmet_r else "Missing resources")
        )
        r_offers.append(
            ResearchOffer(
                research_id=rid,
                research_key=rkey,
                name=str(rinfo.get("name", rkey)),
                current_level=cur,
                target_level=nxt,
                cost=cost,
                can_research=can_r,
                status_text=status_r,
                research_href=f"indexInternal.es?action=internalBuildItem&researchID={rid}",
            )
        )

    planet = PlanetState(
        planet_id=12476,
        name="Homeworld",
        coords="02:380:12",
        planet_type="normal",
        temperature=31,
        fields_used=sum(bld.values()),
        fields_max=230,
        resources=resources,
        storage=Resources(100_000, 100_000, 100_000, 100_000),
        energy=EnergyBalance(produced=e_prod_eff, used=e_used),
        buildings=bld,
        building_names={
            f"b{bid}": str((game_data.BUILDING_BY_ID.get(bid) or {}).get("name", f"b{bid}"))
            for bid in range(1, 15)
            if f"b{bid}" in bld
        },
        ships=shp,
        defense={},
        building_offers=tuple(b_offers),
        ship_offers=tuple(s_offers),
        economy_rates={
            "resource1": 100,
            "resource2": 100,
            "resource3": 100,
            "resource4": 100,
            "resource5": 100,
        },
    )

    raid_targets: tuple[dict[str, Any], ...] = ()
    if include_raid_targets:
        # Place lower historical win-rate target first so ordering cannot bias optimizer results.
        raid_targets = (
            {
                "coords": "02:380:05",
                "owner": "TargetBeta",
                "planet_name": "InactiveOutpostB",
                "status_color": "yellow",
                "is_inactive": True,
                "is_spacia": False,
                "resources": {
                    "pig_iron": 22_000,
                    "metal": 18_000,
                    "kryptonite": 10_000,
                    "spice": 5_000,
                },
                "fleet": {},
                "defense": {},
            },
            {
                "coords": "02:380:04",
                "owner": "TargetAlpha",
                "planet_name": "InactiveFarmA",
                "status_color": "yellow",
                "is_inactive": True,
                "is_spacia": False,
                "resources": {
                    "pig_iron": 22_000,
                    "metal": 18_000,
                    "kryptonite": 10_000,
                    "spice": 5_000,
                },
                "fleet": {},
                "defense": {},
            },
            {
                "coords": "02:380:09",
                "owner": "Spacia",
                "planet_name": "SpaciaSector",
                "status_color": "yellow",
                "is_inactive": True,
                "is_spacia": True,
                "resources": {
                    "pig_iron": 16_000,
                    "metal": 12_000,
                    "kryptonite": 8_000,
                    "spice": 4_000,
                },
                "fleet": {},
                "defense": {"d1": 2},
            },
        )

    return GameState(
        universe_id=31,
        user_id=1001,
        nickname="CommanderDemo",
        timestamp=float(timestamp),
        planets=(planet,),
        active_planet_id=12476,
        researches=res_tech,
        research_names={
            rk: str((game_data.RESEARCH_BY_ID.get(rk) or {}).get("name", rk))
            for rk in res_tech
        },
        specs={"miner": 1, "fleeter": 1},
        research_offers=tuple(r_offers),
        speedup_factor=5.0,
        newbie_protection=False,
        account_points=320,
        account_rank=185,
        raid_targets=raid_targets,
    )


base_ts = 1_700_000_000.0
demo_db = GameDatabase(":memory:")
try:
    for step_idx in range(4):
        snap_ts = base_ts - (4 - step_idx) * 3600.0
        snap_bld = {
            "b1": 4 + (step_idx // 2),
            "b2": 3 + (step_idx // 3),
            "b3": 4 + (step_idx // 2),
            "b4": 2 + (step_idx // 3),
            "b5": 8 + (step_idx // 3),
            "b7": 2 + (step_idx // 3),
            "b8": 2,
            "b9": 2 + (step_idx // 3),
            "b11": 2,
            "b12": 2,
            "b13": 2,
            "b14": 2,
        }
        snap_res = Resources(
            8_000 + step_idx * 1_500,
            5_000 + step_idx * 1_000,
            3_500 + step_idx * 650,
            1_800 + step_idx * 350,
        )
        snap_state = build_sample_game_state(
            timestamp=snap_ts,
            resources=snap_res,
            buildings=snap_bld,
            include_raid_targets=False,
        )
        demo_db.record_state_snapshot(
            snap_state,
            planet_min_interval_seconds=0.0,
            rank_min_interval_seconds=0.0,
        )

    demo_db.record_spy_report(
        SpyReport(
            target_coords="02:380:04",
            owner="TargetAlpha",
            resources=Resources(22_000, 18_000, 10_000, 5_000),
            is_spacia=False,
        ),
        is_inactive=True,
        status_color="yellow",
        ts=base_ts - 900.0,
    )
    demo_db.record_spy_report(
        SpyReport(
            target_coords="02:380:05",
            owner="TargetBeta",
            resources=Resources(22_000, 18_000, 10_000, 5_000),
            is_spacia=False,
        ),
        is_inactive=True,
        status_color="yellow",
        ts=base_ts - 840.0,
    )
    demo_db.record_spy_report(
        SpyReport(
            target_coords="02:380:09",
            owner="Spacia",
            resources=Resources(16_000, 12_000, 8_000, 4_000),
            defense={"d1": 2},
            is_spacia=True,
        ),
        is_inactive=True,
        status_color="yellow",
        ts=base_ts - 780.0,
    )

    sample_actions = [
        ("BUILD", "Pig iron mine", 6, True, "Upgraded mine"),
        ("RESEARCH", "Mining technology", 2, True, "Completed research"),
        ("BUILD_SHIP", "Small fighter", 5, True, "Built raiders"),
        ("THROTTLE_BUILDING", "Blast furnace", 100, True, "Restored smelter"),
        ("ATTACK", "02:380:04", None, True, "Raid dispatched"),
        ("BUILD", "Fusion plant", 9, False, "Queue busy"),
    ]
    for idx, (kind, tgt_name, tgt_lvl, ok, msg) in enumerate(sample_actions):
        demo_db.record_action(
            source="experiment",
            command=kind.lower(),
            action=PlannedAction(
                priority="P2",
                kind=kind,
                planet_id=12476,
                planet_coords="02:380:12",
                target_name=tgt_name,
                target_level=tgt_lvl,
                reason="Historical telemetry seed",
                executable_now=True,
            ),
            executed=ok,
            dry_run=False,
            message=msg,
            ts=base_ts - 6000.0 + idx * 600.0,
        )

    # 02:380:04 -> 9/10 wins; 02:380:05 -> 1/10 fluke with identical last-attack loot;
    # 02:380:09 -> 3/3 Spacia wins.
    shared_last_loot = {"pig_iron": 10_000, "metal": 5_000, "kryptonite": 2_000, "spice": 500}
    for i in range(10):
        ts_04 = base_ts - (10 - i) * 43_200.0
        if i == 4:
            demo_db.record_raid(
                "02:380:04",
                attacked_ts=ts_04,
                is_exempt=True,
                outcome="Defeat / bounced",
                loot={"pig_iron": 0, "metal": 0, "kryptonite": 0, "spice": 0},
            )
        else:
            loot_04 = (
                shared_last_loot
                if i == 9
                else {"pig_iron": 12_000, "metal": 7_500, "kryptonite": 3_500, "spice": 1_200}
            )
            demo_db.record_raid(
                "02:380:04",
                attacked_ts=ts_04,
                is_exempt=True,
                outcome="Victory",
                loot=loot_04,
            )

    for i in range(10):
        ts_05 = base_ts - (10 - i) * 43_200.0 + 300.0
        if i < 9:
            demo_db.record_raid(
                "02:380:05",
                attacked_ts=ts_05,
                is_exempt=True,
                outcome="Defeat / intercepted",
                loot={"pig_iron": 0, "metal": 0, "kryptonite": 0, "spice": 0},
            )
        else:
            demo_db.record_raid(
                "02:380:05",
                attacked_ts=ts_05,
                is_exempt=True,
                outcome="Victory",
                loot=shared_last_loot,
            )

    for i in range(3):
        demo_db.record_raid(
            "02:380:09",
            attacked_ts=base_ts - (3 - i) * 86_400.0,
            is_exempt=True,
            outcome="Victory",
            loot={"pig_iron": 8_000, "metal": 6_000, "kryptonite": 3_000, "spice": 1_500},
        )

    demo_stats = demo_db.get_raid_stats()
    demo_full_summary = demo_db.get_full_history_summary()
    demo_db_counts = dict((demo_db.summary() or {}).get("counts") or {})
finally:
    demo_db.close()

demo_overall = demo_stats["overall"]
demo_per_target = demo_stats["per_target"]
target_rows = []
for coords in sorted(demo_per_target.keys()):
    pt = demo_per_target[coords]
    target_rows.append(
        [
            coords,
            f"{pt['wins']}/{pt['attempts']} ({pt['success_rate']:.0%})",
            f"{pt['total_loot_spice_eq']:,.1f}",
            f"{pt['avg_loot_spice_eq']:,.1f}",
            f"{pt['recency_weighted_loot_spice_eq']:,.1f}",
            pt.get("last_outcome", ""),
            f"{pt.get('last_loot_spice_eq', 0.0):,.1f}",
        ]
    )

print("\nIsolated :memory: Complete Attack History (Overall + Per-Target)")
print(
    f"Overall: {demo_overall['wins']}/{demo_overall['attempts']} wins "
    f"({demo_overall['success_rate']:.1%}), total={demo_overall['total_loot_spice_eq']:,.1f} SpiceEq, "
    f"avg={demo_overall['avg_loot_spice_eq']:,.1f}, recency={demo_overall['recency_weighted_loot_spice_eq']:,.1f}"
)
print(
    render_table(
        ["Target", "Wins/Att (SR)", "Total SpEq", "Avg SpEq", "Recency SpEq", "Last Outcome", "Last SpEq"],
        target_rows,
    )
)
print(
    render_table(
        ["Telemetry Signal", "Summary"],
        [
            ["Table Row Counts", json.dumps(demo_db_counts, sort_keys=True)],
            ["Rank Points Delta", f"+{demo_full_summary['points_delta']:.2f} pts (latest {demo_full_summary['pts_per_hr']:.2f} pts/hr)"],
            ["Economy Trend", f"+{demo_full_summary['economy_trend']:.2f} pts/hr"],
            ["Action History Success Rate", f"{demo_full_summary['action_success_rate']:.1%} ({demo_full_summary['action_successes']}/{demo_full_summary['action_history_count']})"],
            ["Spy Targets Known SpiceEq", f"{demo_full_summary['spy_targets']['total_known_resources_spice_eq']:,.1f} ({demo_full_summary['spy_targets']['count']} targets)"],
        ],
    )
)

## 2. Feature Engineering & Neural Value Network Evaluation

The neural value evaluator encodes each empire state (`GameState` or simulated `_SimState`) via `encode_empire_state(...)` into a fixed `46`-dimensional normalized vector (`STATE_FEATURE_DIM == 46`):
- `[0..15]`: Resource stockpiles, warehouse fill ratios, signed log net hourly production rates, total SpiceEq/h & pts/h, energy margin & deficit fraction.
- `[16..35]`: Planetary mine/power/infrastructure/warehouse levels, free-field ratio, research levels, empire points, fleet power, cargo capacity, queue occupancy, and empirical raid potential.
- `[36..43]`: Complete-history attack features (overall attempts, overall win rate, cumulative loot, blended average/recency loot, last-attack loot & outcome sign, target-level win rate & recency-weighted loot).
- `[44..45]`: SQLite DB telemetry summary features (points/economy trajectory trend and historical action success rate + recon coverage).

Below we compare cold-start vs complete-history feature encodings, generate simulated rollout targets via `simulate_action_sequence`, train `NeuralValueNetwork(backend="auto", seed=42)`, and measure feature sensitivities `dV/dx_j`.

In [ ]:
FEATURE_NAMES: tuple[str, ...] = (
    "stock_pig_iron",
    "stock_metal",
    "stock_kryptonite",
    "stock_spice",
    "fill_pig_iron",
    "fill_metal",
    "fill_kryptonite",
    "fill_spice",
    "rate_pig_iron",
    "rate_metal",
    "rate_kryptonite",
    "rate_spice",
    "rate_spice_eq_hr",
    "rate_pts_hr",
    "min_energy_margin",
    "energy_deficit_frac",
    "lvl_b1_pig",
    "lvl_b2_blast",
    "lvl_b3_kryp",
    "lvl_b4_spice",
    "lvl_power_b5_b6",
    "lvl_b7_dev",
    "lvl_b8_lab_max",
    "lvl_b9_wf",
    "lvl_warehouses",
    "free_fields_ratio",
    "tech_r1_mining",
    "tech_r2_energy",
    "tech_engine_max",
    "tech_r7_spy",
    "tech_combat_sum",
    "empire_points",
    "fleet_power",
    "fleet_cargo",
    "queue_occupancy",
    "raid_potential_and_loot",
    "hist_overall_attempts",
    "hist_overall_win_rate",
    "hist_overall_total_loot",
    "hist_overall_blend_avg_rec",
    "hist_last_loot",
    "hist_last_outcome_sign",
    "hist_target_win_rate",
    "hist_target_recency_loot",
    "tel_points_econ_trend",
    "tel_action_sr_and_recon",
)
assert len(FEATURE_NAMES) == STATE_FEATURE_DIM

base_state_no_hist = build_sample_game_state(timestamp=base_ts)
exp_state = replace(base_state_no_hist, raid_stats=demo_stats)
cold_stats = {
    "overall": {"attempts": 0, "wins": 0, "losses": 0, "success_rate": 0.0},
    "per_target": {},
    "history": [],
}

vec_cold = encode_empire_state(base_state_no_hist, raid_stats=cold_stats)
vec_full = encode_empire_state(exp_state, raid_stats=demo_stats)
assert len(vec_cold) == STATE_FEATURE_DIM and len(vec_full) == STATE_FEATURE_DIM
assert all(math.isfinite(x) for x in vec_cold + vec_full)

feat_cmp_rows = []
for idx in range(35, STATE_FEATURE_DIM):
    delta = vec_full[idx] - vec_cold[idx]
    feat_cmp_rows.append(
        [
            f"[{idx:02d}] {FEATURE_NAMES[idx]}",
            f"{vec_cold[idx]:.4f}",
            f"{vec_full[idx]:.4f}",
            f"{delta:+.4f}",
        ]
    )

print("Cold-Start vs Complete-History Feature Comparison (Indices 35..45)")
print(render_table(["Feature", "Cold-Start", "Full-History", "Delta"], feat_cmp_rows))

eval_cfg = StrategyConfig(
    planner_mode="ga_nn",
    playstyle="fleeter",
    allow_raids=True,
    allow_shipyard=True,
    allow_research=True,
)
candidate_rollouts: list[tuple[str, list[SimAction]]] = [
    ("Idle baseline", []),
    ("Pig mine + Fusion", [SimAction("BUILD", 12476, 1), SimAction("BUILD", 12476, 5)]),
    ("Research Energy + Krypt mine", [SimAction("RESEARCH", 12476, "r2"), SimAction("BUILD", 12476, 3)]),
    ("Shipyard fighters", [SimAction("BUILD_SHIP", 12476, 4, quantity=3)]),
    ("Raid Trap 02:380:05 (10% win)", [SimAction("ATTACK", 12476, target_coords="02:380:05")]),
    ("Raid Farm 02:380:04 (90% win)", [SimAction("ATTACK", 12476, target_coords="02:380:04")]),
    (
        "Combo: Farm + Mine + Fighter",
        [
            SimAction("ATTACK", 12476, target_coords="02:380:04"),
            SimAction("BUILD", 12476, 1),
            SimAction("BUILD_SHIP", 12476, 4, quantity=2),
        ],
    ),
]

rollout_samples: list[tuple[list[float], float]] = []
rollout_table_rows: list[list[Any]] = []
for label, seq in candidate_rollouts:
    sim_res = simulate_action_sequence(exp_state, seq, config=eval_cfg, raid_stats=demo_stats)
    term_vec = encode_empire_state(sim_res.final_state, raid_stats=demo_stats)
    target_val = math.tanh(sim_res.composite_growth_score / 25.0)
    rollout_samples.append((term_vec, target_val))
    rollout_table_rows.append(
        [
            label,
            f"{sim_res.valid_steps}/{len(seq)}",
            f"{sim_res.points_delta:+.2f}",
            f"{sim_res.hourly_spice_eq_delta:+.1f}",
            f"{sim_res.raid_fleet_output_delta:+.2f}",
            f"{sim_res.composite_growth_score:.2f}",
        ]
    )

print("\nForward Simulation Rollout Candidate Scores")
print(
    render_table(
        ["Sequence", "Valid", "Pts Delta", "Income SpEq/h", "Raid/Fleet Delta", "Composite Score"],
        rollout_table_rows,
    )
)

nn_net = create_value_network(backend="auto", seed=42)
pre_cold_v = nn_net.predict(vec_cold)
pre_full_v = nn_net.predict(vec_full)
train_stats = nn_net.train_on_rollouts(rollout_samples, epochs=12, lr=0.025, weight_decay=1e-4)
post_cold_v = nn_net.predict(vec_cold)
post_full_v = nn_net.predict(vec_full)

eps = 1e-4
sensitivities: list[tuple[str, float, float]] = []
for j, fname in enumerate(FEATURE_NAMES):
    xp, xm = list(vec_full), list(vec_full)
    xp[j] += eps
    xm[j] -= eps
    grad_j = (nn_net.predict(xp) - nn_net.predict(xm)) / (2.0 * eps)
    sensitivities.append((fname, vec_full[j], grad_j))

if TORCH_AVAILABLE and torch is not None and nn_net.backend == "torch":
    x_t = torch.tensor(vec_full, dtype=torch.float32, requires_grad=True)
    y_t = nn_net.impl.forward(x_t).squeeze()
    y_t.backward()
    torch_grads = [float(g) for g in x_t.grad.detach().tolist()]
    max_grad_diff = max(abs(a[2] - b) for a, b in zip(sensitivities, torch_grads))
else:
    max_grad_diff = 0.0

top_sens = sorted(sensitivities, key=lambda item: abs(item[2]), reverse=True)[:10]
max_abs_grad = max((abs(g) for _, _, g in top_sens), default=1.0)
sens_rows = [
    [fname, f"{xval:.4f}", f"{grad:+.5f}", ascii_bar(abs(grad), max_abs_grad, width=14)]
    for fname, xval, grad in top_sens
]

print("\nNeural Value Network Training & Top Input Sensitivities")
print(
    render_table(
        ["NN Metric", "Value"],
        [
            ["Backend / Parameters", f"{nn_net.backend} / {nn_net.parameter_count():,} params"],
            ["Rollout Loss (Initial -> Final)", f"{train_stats.initial_loss:.6f} -> {train_stats.final_loss:.6f}"],
            ["Weight Delta Norm", f"{train_stats.weight_delta_norm:.6f}"],
            ["Pred Value Cold -> Full (Pre / Post)", f"({pre_cold_v:+.4f} -> {pre_full_v:+.4f}) / ({post_cold_v:+.4f} -> {post_full_v:+.4f})"],
            ["FD vs Autograd Max Diff", f"{max_grad_diff:.2e}"],
        ],
    )
)
print(render_table(["Feature", "Value x_j", "dV/dx_j", "Magnitude"], sens_rows))

## 3. GA + Neural Network Optimization, Baseline Comparison & Queue ETAs

Next we run `run_optimizer(...)` (and cross-check `optimize_plan_actions(...)`) on the full-history empire scenario:
- Track fitness, points gain, income gain, raid value, and online NN loss across every generation in `OptimizerRunSummary.generation_history`.
- Compare the rule-based `"priority"` baseline against `"ga_nn"` using `plan_actions(...)` (which populates Wait ETA, Duration, Total Completion ETA, and Bottleneck fields via `attach_action_etas`).
- Verify that complete empirical raid history causes `02:380:04` (`90%` win rate) to be prioritized over `02:380:05` (`10%` fluke) despite identical last-attack loot and identical scouted resources.

In [ ]:
prio_cfg = StrategyConfig(
    planner_mode="priority",
    playstyle="fleeter",
    allow_raids=True,
    allow_shipyard=True,
    allow_research=True,
)
ga_cfg = StrategyConfig(
    planner_mode="ga_nn",
    playstyle="fleeter",
    allow_raids=True,
    allow_shipyard=True,
    allow_research=True,
    ga_population_size=16,
    ga_generations=8,
    ga_horizon_steps=5,
    ga_seed=42,
)

summary = run_optimizer(exp_state, config=ga_cfg, raid_stats=demo_stats)
direct_opt_actions = optimize_plan_actions(exp_state, config=ga_cfg, raid_stats=demo_stats)
prio_actions = plan_actions(exp_state, config=prio_cfg, raid_stats=demo_stats)
ga_actions = plan_actions(exp_state, config=ga_cfg, raid_stats=demo_stats)
disk_only_actions = optimize_plan_actions(
    base_state_no_hist, config=ga_cfg, raid_stats=disk_attack_history
)
eta_plans: dict[str, list[PlannedAction]] = {
    "priority": prio_actions,
    "ga_nn": ga_actions,
    "disk_history_ga_nn": attach_action_etas(base_state_no_hist, disk_only_actions),
}

print("Optimizer Run Summary")
print(
    render_table(
        ["Metric", "Value"],
        [
            ["NN Backend / Evaluations", f"{summary.nn_backend} / {summary.nn_evaluations} evals"],
            ["Population / Generations / Seed", f"{summary.population_size} / {summary.generations} / {summary.seed}"],
            ["Baseline -> Best Fitness", f"{summary.baseline_fitness:.3f} -> {summary.best_fitness:.3f} ({summary.improvement_pct:+.2f}%)"],
            ["NN Loss (Initial -> Final)", f"{summary.nn_initial_loss:.6f} -> {summary.nn_final_loss:.6f} (delta={summary.nn_weight_delta_norm:.4f})"],
            ["Best Sim Deltas (Pts / Inc / Raid)", f"{summary.best_simulation.points_delta:+.2f} pts / {summary.best_simulation.hourly_spice_eq_delta:+.1f} SpEq/h / {summary.best_simulation.raid_fleet_output_delta:+.2f}"],
            ["Elapsed Time", f"{summary.elapsed_ms:.2f} ms"],
        ],
    )
)

gen_max_fit = max((g.best_fitness for g in summary.generation_history), default=1.0)
gen_rows = [
    [
        g.generation,
        f"{g.best_fitness:.3f}",
        f"{g.mean_fitness:.3f}",
        f"{g.worst_fitness:.3f}",
        f"{g.best_points_gain:+.2f}",
        f"{g.best_income_gain:+.1f}",
        f"{g.best_raid_value:+.2f}",
        f"{g.nn_loss:.5f}",
        ascii_bar(max(0.0, g.best_fitness), max(1e-6, gen_max_fit), width=14),
    ]
    for g in summary.generation_history
]
print("\nGeneration-by-Generation Progression")
print(
    render_table(
        ["Gen", "Best Fit", "Mean Fit", "Worst Fit", "Pts Gain", "Inc Gain", "Raid Val", "NN Loss", "Curve"],
        gen_rows,
    )
)

if HAS_MATPLOTLIB and plt is not None and summary.generation_history:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    try:
        gens_x = [g.generation for g in summary.generation_history]
        ax.plot(gens_x, [g.best_fitness for g in summary.generation_history], label="Best Fitness")
        ax.plot(gens_x, [g.mean_fitness for g in summary.generation_history], label="Mean Fitness")
        ax.set_xlabel("Generation")
        ax.set_ylabel("Fitness")
        ax.set_title("GA + NN Fitness Progression")
        ax.legend()
        get_be = getattr(matplotlib, "get_backend", None) if matplotlib is not None else None
        backend_name = str(get_be()).lower() if callable(get_be) else ""
        if "inline" in backend_name and hasattr(plt, "show"):
            plt.show()
    finally:
        plt.close(fig)


def _plan_rows(label: str, actions: list[PlannedAction], limit: int = 8) -> list[list[Any]]:
    rows: list[list[Any]] = []
    for idx, act in enumerate(actions[:limit], start=1):
        rows.append(
            [
                label,
                idx,
                act.priority,
                act.kind,
                act.target_name,
                "Y" if act.executable_now else "N",
                act.wait_eta,
                act.duration_text,
                act.total_eta,
                act.eta_bottleneck or "-",
            ]
        )
    return rows


cmp_rows = _plan_rows("priority", prio_actions, limit=6) + _plan_rows("ga_nn", ga_actions, limit=8)
print("\nBaseline ('priority') vs Optimizer ('ga_nn') Action Plan & Queue ETAs")
print(
    render_table(
        ["Mode", "#", "Prio", "Kind", "Target", "Now", "Wait ETA", "Duration", "Total ETA", "Bottleneck"],
        cmp_rows,
    )
)

ga_attack_targets = [a.target_name for a in ga_actions if a.kind == "ATTACK"]
disk_attack_targets = [a.target_name for a in disk_only_actions if a.kind == "ATTACK"]
print(f"GA+NN prioritized attack targets (in execution order): {ga_attack_targets}")
print(
    f"Direct optimize_plan_actions count={len(direct_opt_actions)}, "
    f"disk-only fallback count={len(disk_only_actions)}, "
    f"disk-only attack targets={disk_attack_targets}"
)

bottleneck_probe = PlannedAction(
    priority="P2",
    kind="BUILD",
    planet_id=12476,
    planet_coords="02:380:12",
    target_id=11,
    target_name="Pig iron warehouse",
    target_level=14,
    cost=Resources(50_000, 15_000, 20_000, 5_000),
    missing=Resources(41_000, 8_000, 15_500, 2_800),
    reason="Resource-wait bottleneck ETA inspection",
    executable_now=False,
)
probe_eta = compute_action_eta(exp_state, bottleneck_probe)
print(
    f"Sample resource-wait ETA (Pig iron warehouse L14): wait={probe_eta['wait_eta']}, "
    f"duration={probe_eta['duration_text']}, total={probe_eta['total_eta']}, "
    f"bottleneck={probe_eta['eta_bottleneck']}"
)

## 4. Interactive Hyperparameter Tuning Experiment

Modify `EXPERIMENT_CONFIGS` below to compare alternative playstyles (`"miner"`, `"hybrid"`, `"fleeter"`), GA search budgets (`ga_population_size`, `ga_generations`, `ga_horizon_steps`), neural guidance weights (`ga_nn_weight`), mutation rates (`ga_mutation_rate`), and RNG seeds (`ga_seed`).

In [ ]:
EXPERIMENT_CONFIGS: list[dict[str, Any]] = [
    {
        "label": "miner_default",
        "playstyle": "miner",
        "ga_population_size": 12,
        "ga_generations": 5,
        "ga_horizon_steps": 4,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.25,
        "ga_seed": 42,
    },
    {
        "label": "hybrid_balanced",
        "playstyle": "hybrid",
        "ga_population_size": 12,
        "ga_generations": 5,
        "ga_horizon_steps": 4,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.25,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_pure_ga",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.0,
        "ga_mutation_rate": 0.35,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_nn_035",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.35,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_nn_high_mut",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.65,
        "ga_mutation_rate": 0.40,
        "ga_seed": 42,
    },
    {
        "label": "fleeter_seed_7",
        "playstyle": "fleeter",
        "ga_population_size": 16,
        "ga_generations": 8,
        "ga_horizon_steps": 5,
        "ga_nn_weight": 0.35,
        "ga_mutation_rate": 0.35,
        "ga_seed": 7,
    },
]

sweep_results: list[dict[str, Any]] = []
sweep_rows: list[list[Any]] = []

for spec in EXPERIMENT_CONFIGS:
    sw_cfg = StrategyConfig(
        planner_mode="ga_nn",
        playstyle=str(spec["playstyle"]),
        allow_raids=True,
        allow_shipyard=True,
        allow_research=True,
        ga_population_size=int(spec["ga_population_size"]),
        ga_generations=int(spec["ga_generations"]),
        ga_horizon_steps=int(spec["ga_horizon_steps"]),
        ga_nn_weight=float(spec["ga_nn_weight"]),
        ga_mutation_rate=float(spec["ga_mutation_rate"]),
        ga_seed=int(spec["ga_seed"]),
    )
    sw_sum = run_optimizer(exp_state, config=sw_cfg, raid_stats=demo_stats)
    sw_eta = attach_action_etas(exp_state, sw_sum.actions)
    top_act = sw_eta[0] if sw_eta else None
    top_label = f"{top_act.kind}:{top_act.target_name}" if top_act is not None else "NONE"
    top_total_eta = top_act.total_eta if top_act is not None else "-"
    rec = {
        "label": spec["label"],
        "playstyle": sw_cfg.playstyle,
        "population_size": sw_sum.population_size,
        "generations": sw_sum.generations,
        "horizon_steps": sw_cfg.ga_horizon_steps,
        "nn_weight": sw_cfg.ga_nn_weight,
        "mutation_rate": sw_cfg.ga_mutation_rate,
        "seed": sw_sum.seed,
        "baseline_fitness": sw_sum.baseline_fitness,
        "best_fitness": sw_sum.best_fitness,
        "improvement_pct": sw_sum.improvement_pct,
        "raid_fleet_delta": sw_sum.best_simulation.raid_fleet_output_delta,
        "income_delta": sw_sum.best_simulation.hourly_spice_eq_delta,
        "points_delta": sw_sum.best_simulation.points_delta,
        "top_action": top_label,
        "top_total_eta": top_total_eta,
        "elapsed_ms": sw_sum.elapsed_ms,
    }
    sweep_results.append(rec)
    sweep_rows.append(
        [
            rec["label"],
            rec["playstyle"],
            f"{rec['population_size']}/{rec['generations']}/{rec['horizon_steps']}",
            f"{rec['nn_weight']:.2f}/{rec['mutation_rate']:.2f}/{rec['seed']}",
            f"{rec['baseline_fitness']:.2f} -> {rec['best_fitness']:.2f}",
            f"{rec['improvement_pct']:+.2f}%",
            f"{rec['raid_fleet_delta']:+.2f}",
            f"{rec['top_action']} ({rec['top_total_eta']})",
            f"{rec['elapsed_ms']:.1f}",
        ]
    )

print("Hyperparameter Sweep Results")
print(
    render_table(
        ["Config", "Style", "Pop/Gen/Hor", "NNw/Mut/Seed", "Base -> Best Fit", "Improv", "Raid/Fleet", "Top Action (Total ETA)", "ms"],
        sweep_rows,
    )
)